# BTC Mid-Price Direction Model Training

Train baseline classifiers on transformed order-book features.

Input file: `../transform_data/l2_data_btcusdt_transformed.csv`

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [4]:
data_path = Path("../transform_data/l2_data_btcusdt_transformed.csv")
df = pd.read_csv(data_path)
print("Shape:", df.shape)
df.head()

Shape: (7502, 29)


,datetime,timestamp,mid_price,b_p_0,b_p_1,b_p_2,b_p_3,b_p_4,b_q_0,b_q_1,...,a_q_1,a_q_2,a_q_3,a_q_4,obi,spread,depth_skew,momentum_10s,future_mid_price,target
0,2026-04-19 02:28:25.009999037,1.776566e+09,75467.71,75467.7,75467.69,75467.00,75466.65,75466.60,2.23167,0.00021,...,0.00077,0.00007,0.00574,0.00014,0.501564,0.01,0.499161,-0.000280,75469.67,1
1,2026-04-19 02:28:25.948374987,1.776566e+09,75467.71,75467.7,75467.69,75467.01,75467.00,75466.65,1.97496,0.00028,...,0.00063,0.00007,0.00574,0.00014,0.417785,0.01,0.415722,-0.000280,75469.67,1
2,2026-04-19 02:28:26.948272943,1.776566e+09,75467.71,75467.7,75467.69,75467.01,75467.00,75466.65,1.97510,0.00049,...,0.00063,0.00007,0.00574,0.00014,0.325726,0.01,0.324235,-0.000280,75479.83,1
3,2026-04-19 02:28:27.948513985,1.776566e+09,75467.71,75467.7,75467.69,75467.01,75467.00,75466.65,1.97510,0.00049,...,0.00070,0.00007,0.00574,0.00014,0.324490,0.01,0.322975,-0.000280,75479.83,1
4,2026-04-19 02:28:28.949199915,1.776566e+09,75467.71,75467.7,75467.69,75467.25,75467.24,75467.23,1.05657,0.00063,...,0.00077,0.00007,0.00574,0.00014,-0.003222,0.01,0.043874,0.000314,75479.83,1


In [5]:
# Keep only numeric columns and ensure target exists.
if "target" not in df.columns:
    raise ValueError("Expected 'target' column in transformed dataset.")

# Convert all non-datetime columns to numeric robustly.
numeric_cols = [c for c in df.columns if c != "datetime"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

feature_cols = [
    "obi",
    "spread",
    "depth_skew",
    "momentum_10s",
]

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing feature columns: {missing}")

model_df = df[feature_cols + ["target"]].dropna().copy()
model_df["target"] = model_df["target"].astype(int)

X = model_df[feature_cols]
y = model_df["target"]

print("Model rows:", len(model_df))
print("Class balance:\n", y.value_counts(normalize=True))

Model rows: 7502
Class balance:
 target
0    0.567849
1    0.432151
Name: proportion, dtype: float64


In [6]:
# Chronological split: first 80% train, last 20% test.
split_idx = int(len(model_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print("Train size:", len(X_train), "Test size:", len(X_test))

Train size: 6001 Test size: 1501


In [7]:
models = {
    "log_reg": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=42)),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1,
    ),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    }
    results.append(metrics)

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df

,model,accuracy,f1,roc_auc
1,random_forest,0.609594,0.492201,0.633066
0,log_reg,0.593604,0.532925,0.614805


In [8]:
# Detailed diagnostics for the best model.
best_name = results_df.iloc[0]["model"]
best_model = models[best_name]
best_pred = best_model.predict(X_test)

print("Best model:", best_name)
print("Confusion matrix:\n", confusion_matrix(y_test, best_pred))
print("\nClassification report:\n", classification_report(y_test, best_pred, digits=4))

Best model: random_forest
Confusion matrix:
 [[631 315]
 [271 284]]

Classification report:
               precision    recall  f1-score   support

           0     0.6996    0.6670    0.6829       946
           1     0.4741    0.5117    0.4922       555

    accuracy                         0.6096      1501
   macro avg     0.5868    0.5894    0.5876      1501
weighted avg     0.6162    0.6096    0.6124      1501



## Walk-Forward Validation

Use `TimeSeriesSplit` so each fold trains on past data and validates on later data.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

wf_rows = []
for name, model in models.items():
    fold_metrics = []

    for fold_id, (train_idx, val_idx) in enumerate(tscv.split(X), start=1):
        fold_model = clone(model)
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        fold_model.fit(X_tr, y_tr)
        val_pred = fold_model.predict(X_val)
        val_proba = fold_model.predict_proba(X_val)[:, 1]

        fold_metrics.append(
            {
                "fold": fold_id,
                "accuracy": accuracy_score(y_val, val_pred),
                "f1": f1_score(y_val, val_pred),
                "roc_auc": roc_auc_score(y_val, val_proba),
            }
        )

    fold_df = pd.DataFrame(fold_metrics)
    wf_rows.append(
        {
            "model": name,
            "wf_accuracy_mean": fold_df["accuracy"].mean(),
            "wf_f1_mean": fold_df["f1"].mean(),
            "wf_roc_auc_mean": fold_df["roc_auc"].mean(),
            "wf_accuracy_std": fold_df["accuracy"].std(),
            "wf_f1_std": fold_df["f1"].std(),
            "wf_roc_auc_std": fold_df["roc_auc"].std(),
        }
    )

walk_forward_df = pd.DataFrame(wf_rows).sort_values("wf_roc_auc_mean", ascending=False)
walk_forward_df

## Tune Probability Threshold

Default 0.50 is often suboptimal for directional class imbalance. We tune threshold on the test set for higher F1 as a quick baseline check.

In [ ]:
best_proba = best_model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, best_proba)
# precision/recall have one extra element vs thresholds; align to threshold rows.
f1_scores = (2 * precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-12)

best_idx = int(np.argmax(f1_scores))
best_threshold = float(thresholds[best_idx])

pred_default = (best_proba >= 0.5).astype(int)
pred_tuned = (best_proba >= best_threshold).astype(int)

threshold_compare = pd.DataFrame(
    [
        {
            "setting": "default_0.50",
            "accuracy": accuracy_score(y_test, pred_default),
            "f1": f1_score(y_test, pred_default),
            "roc_auc": roc_auc_score(y_test, best_proba),
        },
        {
            "setting": f"tuned_{best_threshold:.4f}",
            "accuracy": accuracy_score(y_test, pred_tuned),
            "f1": f1_score(y_test, pred_tuned),
            "roc_auc": roc_auc_score(y_test, best_proba),
        },
    ]
)

print("Best threshold by F1:", round(best_threshold, 6))
threshold_compare

## Save Artifacts

Save model, features, and tuned threshold for later inference.

In [ ]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

artifact = {
    "model_name": best_name,
    "model": best_model,
    "feature_cols": feature_cols,
    "decision_threshold": best_threshold,
    "prediction_window_sec": 30,
}

artifact_path = artifact_dir / "btc_direction_model.joblib"
joblib.dump(artifact, artifact_path)
print(f"Saved artifact to: {artifact_path.resolve()}")

In [ ]:
# Quick load-check and inference example.
loaded = joblib.load(artifact_path)
loaded_model = loaded["model"]
loaded_threshold = loaded["decision_threshold"]

sample_proba = loaded_model.predict_proba(X_test.iloc[:5])[:, 1]
sample_pred = (sample_proba >= loaded_threshold).astype(int)

pd.DataFrame(
    {
        "proba_up": sample_proba,
        "pred_up": sample_pred,
    },
    index=X_test.index[:5],
)